# M21 · Convolution, matrices de Toeplitz et équivariance — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m21_convolution/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m21_convolution/03_notebook.ipynb`, créée par `python tools/start_chapter.py M21`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M21.1 à M21.11, M21.17 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M21.19 | La convolution de deux suites, dans les trois modes · `convolve` | 🔨 | ★★ | 30 |
|  | M21.20 | La matrice de Toeplitz d'une convolution · `toeplitz_matrix` | 🔨 | ★★ | 25 |
|  | M21.21 | La convolution circulaire · `circular_convolve` | 🔨 | ★ | 20 |
|  | M21.22 | Reconnaître un noyau séparable et le factoriser · `separable_factors` | 🔨 | ★★ | 25 |
|  | M21.23 | Le noyau équivalent et le pas d'une pile de sommes glissantes · `equivalent_kernel` | 🔨 | ★★ | 30 |
| B | M21.24 | Prédire l'effet d'un décalage de l'entrée sur un filtrage avec pas, puis vérifier | 🔮 | ★ | 15 |
|  | M21.25 | Le coût d'un filtrage direct ou séparable, selon la taille du noyau : mesurer, puis comparer à k² et 2k | 🔬 | ★★ | 30 |
|  | M21.26 | La carte des recouvrements d'une convolution transposée : un damier ou pas, selon k et s | 🎨 | ★ | 15 |
| C | M21.27 | Chasse à l'erreur : un noyau non retourné, des recouvrements supposés réguliers, un noyau de rang 2 « séparé » | 🐛 | ★ | 25 |
|  | M21.28 | Un test de propriété : la matrice de Toeplitz fois x égale la convolution, pour des longueurs tirées au hasard | 🛠️ | ★ | 20 |
| D | M21.29 | Défi : retrouver un noyau inconnu à partir d'un signal et de sa version filtrée bruitée, par les moindres carrés | 🏆 | ★★★ | 25 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M21")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : une loi de somme de dés, des convolutions et des corrélations dans les trois modes, des pochoirs, une matrice de Toeplitz, l'équivariance avec un pas, des positions, une convolution transposée, une carte filtrée, des paramètres et des coûts, un champ récepteur, un pire cas, et un pont.
- Écrire les cinq fonctions de `mymath/convolution.py` (`convolve`, `toeplitz_matrix`, `circular_convolve`, `separable_factors`, `equivalent_kernel`) et les tester.
- Prédire l'effet d'un décalage sur un filtrage avec pas, mesurer le gain d'un noyau séparable, dessiner les recouvrements d'une convolution transposée, corriger trois erreurs classiques, écrire un test de propriété, et relever un défi : retrouver un noyau par les moindres carrés.

**Rappel express.** Convolution $(\mathbf{a} * \mathbf{b})_m = \sum_i a_i\,b_{m-i}$, noyau retourné (`np.convolve`) ; corrélation croisée $(\mathbf{w} \star \mathbf{x})_i = \sum_u w_u\,x_{i+u}$, sans retournement (les bibliothèques) : c'est la convolution valide par $\tilde{\mathbf{w}}$. Modes : plein $n + k - 1$, valide $n - k + 1$, même taille $n$ à partir de l'indice $\lfloor\frac{k-1}{2}\rfloor$ de la pleine. Toeplitz : $T_{ij} = w_{i-j}$, $nk$ coefficients non nuls ; la transposée de la valide est la pleine par $\tilde{\mathbf{w}}$. Circulaire : $\operatorname{circ}(\mathbf{h})^\top = \operatorname{circ}(\check{\mathbf{h}})$, $\check{\mathbf{h}} = (h_0, h_{n-1}, \ldots, h_1)$. Positions : $\lfloor\frac{n + 2p - d(k-1) - 1}{s}\rfloor + 1$. Transposée : $(m - 1)s - 2p + d(k - 1) + 1 + p_{\text{out}}$ valeurs, un damier si $s$ ne divise pas $k$. Paramètres $C_{\text{out}}(\frac{C_{\text{in}}}{g}k_hk_w + 1)$, coût $H_{\text{out}}W_{\text{out}}C_{\text{out}}\frac{C_{\text{in}}}{g}k_hk_w$ ; un noyau de rang 1 se fait en deux passes. Champ récepteur $1 + \sum_\ell d_\ell(k_\ell - 1)s_1\cdots s_{\ell-1}$, pas $S = s_1\cdots s_L$. Pire cas : $\varepsilon\lVert\mathbf{g}\rVert_1$ sur la boule L∞, en $\varepsilon\operatorname{signe}(\mathbf{g})$ ; $\varepsilon\lVert\mathbf{g}\rVert_2$ sur la boule L2.

## Partie 0 · Vérifier tes exercices papier (✏️ M21.1 à M21.11, 🔗 M21.17)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une liste `[0.2, -1]` ; une matrice comme la liste de ses **lignes**, `[[1, 0], [-2, 3]]`, un nombre décimal s'y écrivant avec un point ;
- une **forme** comme un tuple, `(7, 2)`, ou comme un texte, `"(7, 2)"` ;
- une fraction irréductible entre guillemets, `"3/7"`, quand la question le dit ;
- `True` ou `False` pour un vrai ou faux.

Les réponses pas encore remplies affichent ⏳.

**Ex M21.1 — Deux dés truqués et un produit de polynômes : la même convolution**

In [ ]:
# Ex M21.1: the values you found on paper
answer_M21_1a = ...  # a) a list of 6 integers
answer_M21_1b = ...  # b) an irreducible fraction, as text "p/q"
answer_M21_1c = ...  # c) an irreducible fraction, as text "p/q"
answer_M21_1d = ...  # d) an irreducible fraction, as text "p/q"
answer_M21_1e = ...  # e) an integer
answer_M21_1f = ...  # f) an integer
answer_M21_1g = ...  # g) an irreducible fraction, as text "p/q"

for letter, answer in zip("abcdefg", [answer_M21_1a, answer_M21_1b, answer_M21_1c, answer_M21_1d, answer_M21_1e, answer_M21_1f, answer_M21_1g]):
    wbm.check(f"M21.1{letter}", answer)

**Ex M21.2 — Convoluer et corréler un signal à la main, dans les trois modes**

In [ ]:
# Ex M21.2: the values you found on paper
answer_M21_2a = ...  # a) a list of 5 integers
answer_M21_2b = ...  # b) a list of 9 integers
answer_M21_2c = ...  # c) a list of 5 integers
answer_M21_2d = ...  # d) a list of 7 integers
answer_M21_2e = ...  # e) a list of 3 integers
answer_M21_2f = ...  # f) True or False
answer_M21_2g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_M21_2a, answer_M21_2b, answer_M21_2c, answer_M21_2d, answer_M21_2e, answer_M21_2f, answer_M21_2g]):
    wbm.check(f"M21.2{letter}", answer)

**Ex M21.3 — Des pochoirs de différences finies : dérivée et dérivée seconde d'un signal échantillonné**

In [ ]:
# Ex M21.3: the values you found on paper
answer_M21_3a = ...  # a) a list of 5 numbers
answer_M21_3b = ...  # b) a list of 4 numbers
answer_M21_3c = ...  # c) a list of 4 numbers
answer_M21_3d = ...  # d) a number
answer_M21_3e = ...  # e) a list of 2 numbers
answer_M21_3f = ...  # f) True or False
answer_M21_3g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_M21_3a, answer_M21_3b, answer_M21_3c, answer_M21_3d, answer_M21_3e, answer_M21_3f, answer_M21_3g]):
    wbm.check(f"M21.3{letter}", answer)

**Ex M21.4 — La matrice de Toeplitz d'une convolution pleine et d'une convolution valide : coefficients non nuls, rang et noyau**

In [ ]:
# Ex M21.4: the values you found on paper
answer_M21_4a = ...  # a) a matrix (list of rows)
answer_M21_4b = ...  # b) an integer
answer_M21_4c = ...  # c) a list of 7 integers
answer_M21_4d = ...  # d) a matrix (list of rows)
answer_M21_4e = ...  # e) an integer
answer_M21_4f = ...  # f) an integer
answer_M21_4g = ...  # g) True or False
answer_M21_4h = ...  # h) an integer
answer_M21_4i = ...  # i) a list of 3 integers

for letter, answer in zip("abcdefghi", [answer_M21_4a, answer_M21_4b, answer_M21_4c, answer_M21_4d, answer_M21_4e, answer_M21_4f, answer_M21_4g, answer_M21_4h, answer_M21_4i]):
    wbm.check(f"M21.4{letter}", answer)

**Ex M21.5 — Décaler puis filtrer, ou filtrer puis décaler : l'équivariance, ses bords et son pas**

In [ ]:
# Ex M21.5: the values you found on paper
answer_M21_5a = ...  # a) a list of 8 integers
answer_M21_5b = ...  # b) a list of 8 integers
answer_M21_5c = ...  # c) an integer
answer_M21_5d = ...  # d) a list of 4 integers
answer_M21_5e = ...  # e) a list of 4 integers
answer_M21_5f = ...  # f) an integer
answer_M21_5g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M21_5a, answer_M21_5b, answer_M21_5c, answer_M21_5d, answer_M21_5e, answer_M21_5f, answer_M21_5g]):
    wbm.check(f"M21.5{letter}", answer)

**Ex M21.6 — Compter les positions d'une fenêtre : pas, dilatation, padding, « même taille » avec un pas, et le problème inverse**

In [ ]:
# Ex M21.6: the values you found on paper
answer_M21_6a = ...  # a) an integer
answer_M21_6b = ...  # b) an integer
answer_M21_6c = ...  # c) an integer
answer_M21_6d = ...  # d) an integer
answer_M21_6e = ...  # e) an integer
answer_M21_6f = ...  # f) a list of 2 integers
answer_M21_6g = ...  # g) a list of integers
answer_M21_6h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M21_6a, answer_M21_6b, answer_M21_6c, answer_M21_6d, answer_M21_6e, answer_M21_6f, answer_M21_6g, answer_M21_6h]):
    wbm.check(f"M21.6{letter}", answer)

**Ex M21.7 — Sous-échantillonner et insérer des zéros : la convolution transposée d'un signal et ses recouvrements**

In [ ]:
# Ex M21.7: the values you found on paper
answer_M21_7a = ...  # a) a list of 3 integers
answer_M21_7b = ...  # b) a list of 7 integers
answer_M21_7c = ...  # c) an integer
answer_M21_7d = ...  # d) a list of integers
answer_M21_7e = ...  # e) a list of integers
answer_M21_7f = ...  # f) a list of integers
answer_M21_7g = ...  # g) an integer
answer_M21_7h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M21_7a, answer_M21_7b, answer_M21_7c, answer_M21_7d, answer_M21_7e, answer_M21_7f, answer_M21_7g, answer_M21_7h]):
    wbm.check(f"M21.7{letter}", answer)

**Ex M21.8 — Une carte de températures filtrée par un laplacien discret et par un lissage séparable**

In [ ]:
# Ex M21.8: the values you found on paper
answer_M21_8a = ...  # a) a matrix (list of rows)
answer_M21_8b = ...  # b) a list of 2 integers
answer_M21_8c = ...  # c) a list of 2 integers
answer_M21_8d = ...  # d) a 2 × 6 matrix (1 decimal)
answer_M21_8e = ...  # e) a 2 × 4 matrix (2 decimals)
answer_M21_8f = ...  # f) a list of 2 integers
answer_M21_8g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M21_8a, answer_M21_8b, answer_M21_8c, answer_M21_8d, answer_M21_8e, answer_M21_8f, answer_M21_8g]):
    wbm.check(f"M21.8{letter}", answer)

**Ex M21.9 — Paramètres et coût d'un filtrage à plusieurs canaux : standard, groupé, par canal, 1 × 1**

In [ ]:
# Ex M21.9: the values you found on paper
answer_M21_9a = ...  # a) a shape: a tuple (7, 2) or a text "(7, 2)"
answer_M21_9b = ...  # b) an integer
answer_M21_9c = ...  # c) an integer
answer_M21_9d = ...  # d) an integer
answer_M21_9e = ...  # e) an integer
answer_M21_9f = ...  # f) an integer
answer_M21_9g = ...  # g) an integer
answer_M21_9h = ...  # h) a shape: a tuple (7, 2) or a text "(7, 2)"
answer_M21_9i = ...  # i) an integer

for letter, answer in zip("abcdefghi", [answer_M21_9a, answer_M21_9b, answer_M21_9c, answer_M21_9d, answer_M21_9e, answer_M21_9f, answer_M21_9g, answer_M21_9h, answer_M21_9i]):
    wbm.check(f"M21.9{letter}", answer)

**Ex M21.10 — Le noyau équivalent de deux lissages successifs, et le champ récepteur d'une pile avec pas et dilatation**

In [ ]:
# Ex M21.10: the values you found on paper
answer_M21_10a = ...  # a) a list of 4 integers
answer_M21_10b = ...  # b) a list of 4 numbers (4 decimals)
answer_M21_10c = ...  # c) a list of integers
answer_M21_10d = ...  # d) an integer
answer_M21_10e = ...  # e) an integer
answer_M21_10f = ...  # f) an integer
answer_M21_10g = ...  # g) a list of 2 integers
answer_M21_10h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M21_10a, answer_M21_10b, answer_M21_10c, answer_M21_10d, answer_M21_10e, answer_M21_10f, answer_M21_10g, answer_M21_10h]):
    wbm.check(f"M21.10{letter}", answer)

**Ex M21.11 — Le pire cas d'une erreur de capteurs bornée : une forme linéaire sur les boules L∞ et L2**

In [ ]:
# Ex M21.11: the values you found on paper
answer_M21_11a = ...  # a) an integer
answer_M21_11b = ...  # b) a number
answer_M21_11c = ...  # c) a list of 5 numbers
answer_M21_11d = ...  # d) a number (3 decimals)
answer_M21_11e = ...  # e) a number (4 decimals)
answer_M21_11f = ...  # f) a number
answer_M21_11g = ...  # g) a number
answer_M21_11h = ...  # h) a list of 2 numbers

for letter, answer in zip("abcdefgh", [answer_M21_11a, answer_M21_11b, answer_M21_11c, answer_M21_11d, answer_M21_11e, answer_M21_11f, answer_M21_11g, answer_M21_11h]):
    wbm.check(f"M21.11{letter}", answer)

**Ex M21.17 — Pont : filtrer une grille à trois canaux comme les bibliothèques, formes (C, H, W), poids (C_out, C_in, k, k), et sans retournement**

In [ ]:
# Ex M21.17: the values you found on paper
answer_M21_17a = ...  # a) a shape: a tuple (7, 2) or a text "(7, 2)"
answer_M21_17b = ...  # b) a shape: a tuple (7, 2) or a text "(7, 2)"
answer_M21_17c = ...  # c) a shape: a tuple (7, 2) or a text "(7, 2)"
answer_M21_17d = ...  # d) an integer
answer_M21_17e = ...  # e) an integer
answer_M21_17f = ...  # f) an integer
answer_M21_17g = ...  # g) a 2 × 2 matrix (list of rows)
answer_M21_17h = ...  # h) an integer
answer_M21_17i = ...  # i) a list of 5 integers
answer_M21_17j = ...  # j) a list of 5 integers

for letter, answer in zip("abcdefghij", [answer_M21_17a, answer_M21_17b, answer_M21_17c, answer_M21_17d, answer_M21_17e, answer_M21_17f, answer_M21_17g, answer_M21_17h, answer_M21_17i, answer_M21_17j]):
    wbm.check(f"M21.17{letter}", answer)

## Partie A · Ta librairie : la convolution, sa matrice, la convolution circulaire, les noyaux séparables et le noyau équivalent

Fiche §1 à §4, §6 et §7. Les cinq fonctions de `mymath/convolution.py`, leurs tests, et un usage de chacune.

In [ ]:
# Tools of the notebook (parts A to D)
import itertools
import math
import re
import time
from fractions import Fraction

import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg
import scipy.signal

from wbm.errors import MymathMissingError
from wbm.nbtools import StopCheck, checked, drawn, filled, run_pytest_group, same, verdict

TEST_M21 = "tests/test_m21_convolution.py"            # the tests of mymath/convolution.py


def shown(value, digits=4):
    """A short text for a result of the learner's code (the check messages): numbers rounded, as lists."""
    if value is None:
        return "None"
    try:
        if isinstance(value, tuple):
            return "(" + ", ".join(str(np.round(np.asarray(v, dtype=float), digits).tolist()) for v in value) + ")"
        return str(np.round(np.asarray(value, dtype=float), digits).tolist())
    except (TypeError, ValueError):
        return repr(value)[:120]

### Ex M21.19 — La convolution de deux suites, dans les trois modes · `convolve` 🔨 ★★ ⏱️ 30 min
**Objectif :** programmer la convolution des mathématiciens dans les trois modes, puis s'en servir pour la loi d'une somme de dés, une puissance d'un polynôme et une moyenne mobile.  
**Prérequis :** fiche §1 et §2 ; R3, R5, M02b · **mymath :** `convolution.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/convolution.py`, lis la docstring du module puis celle de la fonction, remplace chaque `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. ⚠️ Le squelette n'importe rien : ajoute `import numpy as np` en tête du fichier (et `import numbers` si tu t'en sers). Ni `np.convolve`, ni `np.correlate`, ni `np.polymul`, ni `np.polynomial`, ni `np.fft`, aucune fonction de SciPy, et, de `np.linalg`, seulement `svd`, dans `separable_factors` : les tests comparent tes résultats aux leurs, et un test lit ton module. Les suites et les matrices se rendent en tableaux NumPy de flottants, les entiers en `int` de Python ; ne modifie jamais les tableaux reçus.

Écris `convolve(a, b, mode="full")` (fiche §1 et §2), en suivant sa docstring : la convolution des mathématiciens, $(\mathbf{a} * \mathbf{b})_m = \sum_i a_i\,b_{m-i}$, qui retourne le noyau, comme `np.convolve`. Le mode `"full"` rend les $n + m - 1$ valeurs ; `"valid"` les indices $\min(n, m) - 1$ à $\max(n, m) - 1$ de la convolution pleine ; `"same"` ses $\max(n, m)$ valeurs à partir de l'indice $\big\lfloor\frac{\min(n, m) - 1}{2}\big\rfloor$ (la convention de NumPy). Un tableau de flottants ; `ValueError` pour une suite qui n'est pas à une dimension ou qui est vide, ou un mode inconnu.

**L'usage.** Avec ta fonction :
- `law_19` : la loi de la somme de trois dés équilibrés à 8 faces, numérotées de 1 à 8, lancés indépendamment, rangée par **valeur de la somme** : un dé est la suite $(0, \frac18, \ldots, \frac18)$ de 9 termes (la case 0, de probabilité 0, puis les faces 1 à 8), et la loi de la somme, convoluée trois fois, a 25 termes, l'indice étant la somme (de 0 à 24) ;
- a) la probabilité que les trois dés totalisent 11, en fraction irréductible (`"p/q"` ; la valeur flottante de `law_19[11]`, exacte ici, est aussi acceptée) ;
- b) le coefficient de $X^5$ dans $(2 - X)^7$, par sept convolutions successives (les coefficients d'un polynôme rangés du terme constant au terme de plus haut degré) ;
- c) la première valeur du mode même taille de la moyenne mobile sur 5 jours (`AVERAGE_19`, cinq valeurs égales à 0,2) des températures `TEMPERATURES_19` (la valeur que donne le calcul, sans l'arrondir) ;
- d) le nombre de valeurs de ce mode même taille qui sont des moyennes de 5 vraies mesures.

La vérification compare aussi `law_19` à la loi calculée par `np.convolve`. Dans ta copie : pourquoi la case 0 du dé fait-elle de l'indice la valeur de la somme ? Quelles valeurs du mode même taille sont tirées vers 0 par les zéros ajoutés, et pourquoi ?

In [ ]:
TEMPERATURES_19 = np.array([8.0, 10, 13, 15, 14, 12, 11, 9, 7, 6, 8, 10])     # 12 daily temperatures (°C)
AVERAGE_19 = np.full(5, 0.2)                                                  # the moving average over 5 days

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M21")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M21, "test_convolve_ or test_module_uses_")

**L'usage** : la loi d'une somme de dés, une puissance d'un polynôme, une moyenne mobile et ses bords.

In [ ]:
law_19 = answer_M21_19a = answer_M21_19b = answer_M21_19c = answer_M21_19d = ...
with wbm.attempt("M21.19"):          # if you call convolve here: ⏳ while it is not written
    law_19 = ...          # the 25 probabilities of the sum of three 8-sided dice, the index being the sum (0 to 24)
    answer_M21_19a = ...  # a) P(sum = 11): an irreducible fraction "p/q" (or the exact float law_19[11])
    answer_M21_19b = ...  # b) an integer: the coefficient of X^5 in (2 - X)^7, by seven successive convolutions
    answer_M21_19c = ...  # c) a number, as computed (not rounded): the first value of the "same" mode of the average
    answer_M21_19d = ...  # d) an integer: how many values of this "same" mode are averages of 5 true measures

In [ ]:
with checked("M21.19"):
    if filled(law_19):
        die_c19 = np.r_[0.0, np.full(8, 1 / 8)]
        same("M21.19", "law_19, la loi de la somme de trois dés à 8 faces (indice = somme)", law_19,
             np.convolve(np.convolve(die_c19, die_c19), die_c19))
    else:
        print("⏳ Ex M21.19 : la comparaison attend ton tableau law_19.")
for letter, answer in zip("abc", [answer_M21_19a, answer_M21_19b, answer_M21_19c]):
    wbm.check(f"M21.19{letter}", answer, computed=True)
wbm.check("M21.19d", answer_M21_19d)

### Ex M21.20 — La matrice de Toeplitz d'une convolution · `toeplitz_matrix` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer la matrice de Toeplitz d'une convolution, puis lire ses rangs, sa proportion de coefficients non nuls et sa transposée.  
**Prérequis :** fiche §3 ; M12a ; Ex M21.4 · **mymath :** `convolution.py` · **Parcours :** C, E

> Même mode d'emploi qu'en M21.19 : écris la fonction dans ta copie `mon_travail/mymath/convolution.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `toeplitz_matrix(kernel, n, mode="full")` (fiche §3), en suivant sa docstring : la matrice $\mathbf{T}$ de $\mathbf{x} \mapsto$ `convolve(x, kernel, mode)` pour $\mathbf{x}$ de longueur $n$. En mode plein, sa colonne $j$ est le noyau descendu de $j$ lignes, $T_{ij} = w_{i-j}$, de forme $(n + k - 1, n)$ ; les autres modes gardent les mêmes lignes que `convolve` ($n - k + 1$ en mode valide, $n$ en mode même taille, qui demandent $n \ge k$). Un tableau de flottants ; `ValueError` pour un noyau qui n'est pas à une dimension ou qui est vide, un $n$ qui n'est pas un entier $\ge 1$ (un booléen est refusé), $n < k$ en modes valide et même taille, ou un mode inconnu.

**L'usage.** La cellule fournie donne le noyau $(1, -3, 2)$ de M21.4 (`W4_20`), un noyau de 7 coefficients non nuls (`W7_20`), et deux noyaux pour l'énoncé de la transposée (fiche §3) : $\mathbf{w} = (1, 4, -2, 3)$, de longueur paire (`W_EVEN_20`), et $(2, 0, -1, 5, 1)$, de longueur impaire (`W_ODD_20`). Avec ta fonction :
- a) les rangs des matrices pleine et valide de `W4_20` pour $n = 6$, dans cet ordre (`np.linalg.matrix_rank` ; une liste de 2 entiers) ;
- b) le pourcentage de coefficients non nuls de la matrice pleine de `W7_20` pour $n = 1\,000$ (la valeur que donne le calcul, sans l'arrondir) ;
- c) trois vrai ou faux, dans cet ordre (`np.array_equal`) : la transposée de la matrice valide de `W_EVEN_20` pour $n = 7$ est la matrice pleine de son noyau retourné $\tilde{\mathbf{w}}$ (pour $n = 4$) ; la transposée de sa matrice même taille ($n = 7$) est la matrice même taille de $\tilde{\mathbf{w}}$ ($n = 7$) ; la même chose que la deuxième pour `W_ODD_20`.

Dans ta copie : quels signaux la matrice valide de a) ne voit-elle pas ? Que devient le pourcentage de b) quand $n$ double ? Pour c), pourquoi la transposition échange-t-elle valide et plein, et pourquoi ne garde-t-elle le mode même taille que pour un noyau de longueur impaire ?

In [ ]:
W4_20 = np.array([1.0, -3.0, 2.0])                                  # the kernel of M21.4
W7_20 = np.array([4.0, -2.0, 1.0, 3.0, -1.0, 2.0, 5.0])                 # a kernel of 7 non-zero coefficients
W_EVEN_20 = np.array([1.0, 4.0, -2.0, 3.0])                              # k = 4
W_ODD_20 = np.array([2.0, 0.0, -1.0, 5.0, 1.0])                          # k = 5

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M21")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M21, "test_toeplitz_matrix_")

**L'usage** : des rangs, une matrice creuse, et l'énoncé de la transposée.

In [ ]:
answer_M21_20a = answer_M21_20b = answer_M21_20c = ...
with wbm.attempt("M21.20"):          # if you call toeplitz_matrix here: ⏳ while it is not written
    answer_M21_20a = ...  # a) a list of 2 integers: the ranks of the full, then valid, matrix of W4_20 (n = 6)
    answer_M21_20b = ...  # b) a number, as computed (not rounded): the % of non-zero entries, full matrix of W7_20, n = 1000
    answer_M21_20c = ...  # c) a list of 3 True or False: valid (W_EVEN_20) vs full; same vs same (W_EVEN_20); same vs same (W_ODD_20)

In [ ]:
for letter, answer in zip("abc", [answer_M21_20a, answer_M21_20b, answer_M21_20c]):
    wbm.check(f"M21.20{letter}", answer, computed=True)

### Ex M21.21 — La convolution circulaire · `circular_convolve` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer la convolution circulaire, puis lisser un profil journalier périodique et vérifier la transposée d'une matrice circulante.  
**Prérequis :** fiche §4 · **mymath :** `convolution.py` · **Parcours :** C

> Même mode d'emploi qu'en M21.19 : écris la fonction dans ta copie `mon_travail/mymath/convolution.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `circular_convolve(a, b)` (fiche §4), en suivant sa docstring : la convolution circulaire de deux suites de même longueur $n$, $(\mathbf{a} \circledast \mathbf{b})_m = \sum_i a_i\,b_{(m-i) \bmod n}$, les indices lus modulo $n$. Un tableau de flottants ; `ValueError` pour une suite qui n'est pas à une dimension ou qui est vide, ou deux longueurs différentes.

**L'usage.** La cellule fournie donne le profil moyen d'une journée, une température (°C) par heure de 0 h à 23 h (`PROFILE_21`) ; l'heure 24 est l'heure 0. On le lisse par la moyenne circulaire centrée sur 3 heures, le noyau $\mathbf{h}$ de 24 valeurs nulles sauf $h_0 = h_1 = h_{23} = \frac13$ (`H_21`). Avec ta fonction :
- `smooth_21` : les 24 températures lissées, la convolution circulaire de `PROFILE_21` par `H_21` ;
- a) la valeur lissée à 0 h (la valeur que donne le calcul, sans l'arrondir) ;
- b) l'heure du maximum lissé (un entier) ;
- c) vrai ou faux : en complétant $(3, -1, 2, 5)$ et $(1, 2, -2)$ (`A_21`, `B_21`) par des zéros jusqu'à la longueur 6, la convolution circulaire donne la convolution pleine (compare à ta `convolve`) ;
- d) pour $\mathbf{h} = (5, -2, 0, 3, 1, 4)$ (`HC_21`), en construisant $\operatorname{circ}(\mathbf{h})$ colonne par colonne avec ta fonction (la colonne $j$ est la convolution circulaire du $j$-ième vecteur de la base canonique par $\mathbf{h}$) : deux vrai ou faux, dans cet ordre, $\operatorname{circ}(\mathbf{h})^\top = \operatorname{circ}(\check{\mathbf{h}})$, puis $\operatorname{circ}(\mathbf{h})^\top$ égale la matrice circulante du simple renversement de $\mathbf{h}$.

La vérification compare aussi `smooth_21` au produit de la matrice circulante de SciPy par le profil. Dans ta copie : quelles heures entrent dans la valeur lissée de 0 h, et que donnerait une convolution ordinaire en mode même taille ? Écris $\check{\mathbf{h}}$, et le renversement de $\mathbf{h}$ : de combien de cases sont-ils décalés ?

In [ ]:
PROFILE_21 = np.array([9.0, 8, 8, 7, 6, 7, 8, 10, 12, 14, 16, 17,
                       19, 20, 21, 20, 18, 16, 14, 13, 12, 11, 10, 9])   # °C, from 0 h to 23 h; hour 24 is hour 0
H_21 = np.zeros(24)
H_21[[0, 1, 23]] = 1 / 3                                     # the centred circular average over 3 hours
A_21, B_21 = np.array([3.0, -1.0, 2.0, 5.0]), np.array([1.0, 2.0, -2.0])
HC_21 = np.array([5.0, -2.0, 0.0, 3.0, 1.0, 4.0])

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M21")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M21, "test_circular_convolve_")

**L'usage** : lisser un profil périodique, éviter le repliement, et la transposée d'une matrice circulante.

In [ ]:
smooth_21 = answer_M21_21a = answer_M21_21b = answer_M21_21c = answer_M21_21d = ...
with wbm.attempt("M21.21"):          # if you call circular_convolve here: ⏳ while it is not written
    smooth_21 = ...       # the 24 smoothed temperatures: the circular convolution of PROFILE_21 by H_21
    answer_M21_21a = ...  # a) a number, as computed (not rounded): the smoothed value at 0 h
    answer_M21_21b = ...  # b) an integer: the hour of the largest smoothed value
    answer_M21_21c = ...  # c) True or False: A_21 and B_21 padded with zeros to length 6, circular = full convolution
    answer_M21_21d = ...  # d) a list of 2 True or False: circ(h)ᵀ = circ(ȟ), then circ(h)ᵀ = circ(h reversed)

In [ ]:
with checked("M21.21"):
    if filled(smooth_21):
        same("M21.21", "smooth_21, comparé à scipy.linalg.circulant(H_21) @ PROFILE_21", smooth_21,
             scipy.linalg.circulant(H_21) @ PROFILE_21)
    else:
        print("⏳ Ex M21.21 : la comparaison attend ton tableau smooth_21.")
for letter, answer in zip("abcd", [answer_M21_21a, answer_M21_21b, answer_M21_21c, answer_M21_21d]):
    wbm.check(f"M21.21{letter}", answer, computed=True)

### Ex M21.22 — Reconnaître un noyau séparable et le factoriser · `separable_factors` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer la factorisation d'un noyau de rang 1 par la décomposition en valeurs singulières, et voir ce qu'en dit Eckart-Young sur un noyau presque séparable.  
**Prérequis :** fiche §6 ; M12d ; Ex M21.8 · **mymath :** `convolution.py` · **Parcours :** C

> Même mode d'emploi qu'en M21.19 : écris la fonction dans ta copie `mon_travail/mymath/convolution.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `separable_factors(kernel, tol=1e-10)` (fiche §6), en suivant sa docstring : par la décomposition en valeurs singulières (M12d, `np.linalg.svd`, la seule fonction de `np.linalg` permise, ici seulement), le noyau est séparable quand sa deuxième valeur singulière vaut au plus `tol` fois la première ; alors $\mathbf{v}$ est le premier vecteur singulier à droite, de norme 1, sa première coordonnée de valeur absolue $> 10^{-8}$ rendue positive, et $\mathbf{u} = \mathbf{K}\mathbf{v}$ ; sinon, `None`. `ValueError` pour un noyau qui n'est pas à deux dimensions, vide ou identiquement nul, ou une tolérance négative.

**L'usage.** La cellule fournie donne le lissage $\mathbf{K} = \mathbf{a}\mathbf{a}^\top$ de M21.8, avec $\mathbf{a} = \frac15(1, 3, 1)$ (`K8_22`), la moyenne en croix $\frac15\begin{pmatrix} 0 & 1 & 0 \\ 1 & 1 & 1 \\ 0 & 1 & 0 \end{pmatrix}$ (`CROSS_22`), et un noyau presque séparable, le produit extérieur de $(1, 2, 4, 2, 1)$ et de $(2, -1, 3)$ plus un bruit gaussien d'écart type 0,001 (`NOISY_22`, graine 5532). Avec ta fonction :
- a) le facteur $\mathbf{v}$ de `K8_22` (une liste de 3 nombres, la valeur que donne le calcul, sans l'arrondir) ;
- b) le rapport $s_2/s_1$ des valeurs singulières de `CROSS_22` (`np.linalg.svd` ; la valeur que donne le calcul, sans l'arrondir), et `factors_cross_22`, ce que rend ta fonction pour ce noyau ;
- `factors_default_22`, ce que rend ta fonction pour `NOISY_22` avec la tolérance par défaut, et `factors_loose_22`, avec une tolérance de $10^{-2}$ ;
- c) l'erreur $\lVert\mathbf{K} - \mathbf{u}\mathbf{v}^\top\rVert_F$ de l'approximation de rang 1 donnée par `factors_loose_22` (`np.linalg.norm` ; la valeur que donne le calcul, sans l'arrondir).

La vérification regarde aussi que `factors_cross_22` et `factors_default_22` valent `None`, que `factors_loose_22` est un couple de facteurs, et compare c) à $\sqrt{s_2^2 + s_3^2}$, calculé par NumPy. Dans ta copie : que vaut $\mathbf{u}$ en a) ? Quelles sont les valeurs singulières de la moyenne en croix, et pourquoi son rang vaut-il 2 ? Pourquoi c) vaut-il $\sqrt{s_2^2 + s_3^2}$ (Eckart-Young, M12d) ? Quel rapport $\frac{s_2}{s_1}$ sépare les deux tolérances ?

In [ ]:
A8_22 = np.array([1.0, 3.0, 1.0]) / 5
K8_22 = np.outer(A8_22, A8_22)                                       # the smoothing kernel of M21.8
CROSS_22 = np.array([[0.0, 1, 0], [1, 1, 1], [0, 1, 0]]) / 5         # the mean over a cross of five cells
NOISY_22 = (np.outer([1.0, 2.0, 4.0, 2.0, 1.0], [2.0, -1.0, 3.0])
            + 1e-3 * np.random.default_rng(5532).standard_normal((5, 3)))   # nearly separable

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M21")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M21, "test_separable_factors_")

**L'usage** : un noyau séparable, un noyau qui ne l'est pas, et un noyau presque séparable.

In [ ]:
answer_M21_22a = answer_M21_22b = answer_M21_22c = ...
factors_cross_22 = factors_default_22 = factors_loose_22 = ...
with wbm.attempt("M21.22"):          # if you call separable_factors here: ⏳ while it is not written
    answer_M21_22a = ...      # a) a list of 3 numbers, as computed (not rounded): the factor v of K8_22
    answer_M21_22b = ...      # b) a number, as computed: s2 / s1 for CROSS_22 (np.linalg.svd)
    factors_cross_22 = ...    # what separable_factors(CROSS_22) returns
    factors_default_22 = ...  # separable_factors(NOISY_22), with the default tolerance
    factors_loose_22 = ...    # separable_factors(NOISY_22, tol=1e-2)
    answer_M21_22c = ...      # c) a number, as computed: the Frobenius norm of NOISY_22 - outer(u, v), (u, v) = factors_loose_22

In [ ]:
for letter, answer in zip("abc", [answer_M21_22a, answer_M21_22b, answer_M21_22c]):
    wbm.check(f"M21.22{letter}", answer, computed=True)
if factors_cross_22 is not ...:                           # None is the right answer here: not filled() (None = ⏳)
    verdict("M21.22", factors_cross_22 is None, "la moyenne en croix, de rang 2, n'est pas séparable : None.",
            "la moyenne en croix est de rang 2 (s₂ n'est pas petit devant s₁) : ta fonction doit rendre None.")
if factors_default_22 is not ...:
    verdict("M21.22", factors_default_22 is None, "avec la tolérance par défaut, NOISY_22 n'est pas séparable.",
            "avec la tolérance par défaut (1e-10), NOISY_22 a s₂ trop grand devant s₁ : ta fonction doit rendre None.")
if filled(factors_loose_22):
    verdict("M21.22", isinstance(factors_loose_22, tuple) and len(factors_loose_22) == 2,
            "avec la tolérance 1e-2, NOISY_22 est accepté : un couple (u, v).",
            "avec la tolérance 1e-2, ta fonction doit rendre le couple (u, v).")
if filled(answer_M21_22c):
    singular_c22 = np.linalg.svd(NOISY_22, compute_uv=False)
    verdict("M21.22", abs(answer_M21_22c - math.sqrt(np.sum(singular_c22[1:] ** 2))) < 1e-12,
            "l'erreur vaut √(s₂² + s₃²), comme le dit Eckart-Young.",
            "l'erreur de la meilleure approximation de rang 1 doit valoir √(s₂² + s₃²) : relis tes facteurs.")

### Ex M21.23 — Le noyau équivalent et le pas d'une pile de sommes glissantes · `equivalent_kernel` 🔨 ★★ ⏱️ 30 min
**Objectif :** programmer le noyau équivalent et le pas total d'une pile de sommes glissantes, et le vérifier en appliquant la pile étage par étage.  
**Prérequis :** fiche §7 ; Ex M21.10 et M21.19 · **mymath :** `convolution.py` · **Parcours :** C, E

> Même mode d'emploi qu'en M21.19 : écris la fonction dans ta copie `mon_travail/mymath/convolution.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `equivalent_kernel(kernels, strides=None)` (fiche §7), en suivant sa docstring, avec ta `convolve` : l'étage $\ell$ calcule la somme glissante des bibliothèques $y_i = \sum_u w_{\ell,u}\,x_{s_\ell i + u}$ (sans retournement ni padding), et toute la pile est une seule somme glissante $z_j = \sum_t K_t\,x_{Sj+t}$, de pas $S = s_1 \cdots s_L$. En partant de $\mathbf{K} = (1)$ et $S = 1$, chaque étage remplace $\mathbf{K}$ par sa convolution avec $\mathbf{w}_\ell$ **dilaté** de $S$ ($S - 1$ zéros entre deux coefficients), puis $S$ par $S\,s_\ell$. Le couple `(K, S)` : un tableau de flottants, de longueur $1 + \sum_\ell (k_\ell - 1)\,s_1 \cdots s_{\ell-1}$ (zéros compris), et un `int` ; `ValueError` sans noyau, pour un noyau qui n'est pas à une dimension ou qui est vide, ou des pas qui ne sont pas $L$ entiers $\ge 1$.

**L'usage.** Écris aussi `strided_sum(x, w, s)`, la somme glissante de pas $s$ en mode valide, avec ta `convolve` (une somme glissante est une convolution valide par le noyau retourné, dont on garde une valeur sur $s$). La cellule fournie tire (graine 5533) une pile de trois étages, de noyaux de longueurs 3, 4 et 2 (`KERNELS_23`) et de pas 2, 3 et 1 (`STRIDES_23`), et un signal de 200 valeurs (`X_23`). Avec tes fonctions :
- a) $6\,\mathbf{K}$ pour la pile de M21.10 c) : $\frac12(1, 1)$ avec un pas de 3, puis $\frac13(1, 1, 1)$ avec un pas de 1 (une liste de nombres, la valeur que donne le calcul) ;
- b) la longueur du noyau équivalent et le pas total de quatre étages identiques, de noyaux de longueur 4 et de pas 2 (une liste de 2 entiers) ;
- c) le champ récepteur de la pile de M21.10 e), $(k, s, d) = (2, 2, 1)$, $(5, 4, 1)$, $(3, 1, 4)$, en écrivant le troisième étage, dilaté, comme le noyau à trous $(w_0, 0, 0, 0, w_1, 0, 0, 0, w_2)$ (un entier : la longueur du noyau équivalent) ;
- `stack_23` : `X_23` passé dans les trois étages, l'un après l'autre, par `strided_sum` ; `direct_23` : `X_23` passé dans la seule somme glissante du noyau équivalent, avec le pas total ;
- d) vrai ou faux : `stack_23` et `direct_23` coïncident à $10^{-12}$ près sur leurs positions communes (`np.allclose(..., rtol=0, atol=1e-12)`).

La vérification essaie aussi `strided_sum` sur des cas que tu ne vois pas, et compare `stack_23` et `direct_23` à la pile calculée par `np.correlate`. Dans ta copie : pourquoi le noyau de c) a-t-il des coefficients nuls, et pourquoi faut-il les garder ? Combien de positions ont `stack_23` et `direct_23`, et pourquoi la seconde peut-elle en avoir plus ?

In [ ]:
rng_23 = np.random.default_rng(5533)
KERNELS_23 = [rng_23.standard_normal(k) for k in (3, 4, 2)]     # a random stack: kernels of lengths 3, 4 and 2
STRIDES_23 = [2, 3, 1]                                         # and steps 2, 3 and 1
X_23 = rng_23.standard_normal(200)                             # a signal of 200 values

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M21")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M21, "test_equivalent_kernel_")

**L'usage** : trois piles, et une pile tirée au hasard, appliquée de deux façons.

In [ ]:
def strided_sum(x, w, s=1):
    """The sliding sum of the libraries with the step s, valid mode: y[i] = Σ_u w[u] x[s i + u] for every i whose
    window fits (with your convolve)."""
    raise NotImplementedError("strided_sum n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M21_23a = answer_M21_23b = answer_M21_23c = answer_M21_23d = stack_23 = direct_23 = ...
with wbm.attempt("M21.23"):          # equivalent_kernel and strided_sum: ⏳ while they are not written
    answer_M21_23a = ...  # a) a list of numbers, as computed: 6 K for (1, 1)/2 with a step 3, then (1, 1, 1)/3 with a step 1
    answer_M21_23b = ...  # b) a list of 2 integers: [len(K), S] for four stages of kernels of length 4 and step 2
    answer_M21_23c = ...  # c) an integer: len(K) for the stack of M21.10 e), its third stage written with its holes
    stack_23 = ...        # X_23 through the three stages of KERNELS_23 and STRIDES_23, one after the other
    direct_23 = ...       # X_23 through the single sliding sum of the equivalent kernel, with the total step
    answer_M21_23d = ...  # d) True or False: they coincide to 1e-12 on their common positions

In [ ]:
for letter, answer in zip("abcd", [answer_M21_23a, answer_M21_23b, answer_M21_23c, answer_M21_23d]):
    wbm.check(f"M21.23{letter}", answer, computed=True)

with checked("M21.23"):
    rng_c23 = np.random.default_rng(5523)
    wrong_23 = []
    for n_c23, k_c23, s_c23 in ((17, 4, 3), (12, 1, 2), (9, 9, 4), (20, 3, 1)):
        x_c23, w_c23 = rng_c23.standard_normal(n_c23), rng_c23.standard_normal(k_c23)
        want_c23 = np.correlate(x_c23, w_c23, "valid")[::s_c23]
        got_c23 = strided_sum(x_c23.copy(), w_c23.copy(), s_c23)
        if got_c23 is None or np.shape(got_c23) != want_c23.shape or not np.allclose(got_c23, want_c23):
            wrong_23.append(f"n = {n_c23}, k = {k_c23}, s = {s_c23} : attendu {np.round(want_c23, 4).tolist()}, "
                            f"reçu {shown(got_c23)}"[:260])
    verdict("M21.23", not wrong_23, "strided_sum : juste sur quatre cas que tu ne vois pas (noyaux de longueur 1 à 9).",
            "strided_sum, " + " ; ".join(wrong_23[:2]))
    if wrong_23:
        raise StopCheck
    if filled(stack_23, direct_23):
        want_stack_23 = X_23
        for w_c23, s_c23 in zip(KERNELS_23, STRIDES_23):
            want_stack_23 = np.correlate(want_stack_23, w_c23, "valid")[::s_c23]
        same("M21.23", "stack_23, la pile appliquée étage par étage", stack_23, want_stack_23)
        common_23 = min(len(stack_23), len(direct_23))
        same("M21.23", "direct_23, le noyau équivalent et le pas total (positions communes)",
             np.asarray(direct_23, dtype=float)[:common_23], want_stack_23[:common_23])
    else:
        print("⏳ Ex M21.23 : la comparaison attend tes tableaux stack_23 et direct_23.")

## Partie B · Prédire, mesurer, dessiner

Fiche §4 à §6. Un décalage de l'entrée d'un filtrage avec pas, prédit puis vu ; le coût d'un filtrage direct ou séparable, mesuré ; les recouvrements d'une convolution transposée, dessinés.

### Ex M21.24 — Prédire l'effet d'un décalage de l'entrée sur un filtrage avec pas, puis vérifier 🔮 ★ ⏱️ 15 min
**Objectif :** prévoir ce qu'un décalage de l'entrée fait à un filtrage avec pas, puis le voir.  
**Prérequis :** fiche §4 et §5 ; Ex M21.19 · **Parcours :** C, E

Un signal de 30 valeurs, nul sauf $x_7 = 2$, $x_8 = 5$ et $x_9 = -1$ (`X_24`), est filtré par la somme glissante de poids $(1, -1, 2, 1)$ (`W_24`) avec un pas de 3, en mode valide : $y_i = \sum_u w_u\,x_{3i+u}$, pour les 9 positions $i = 0, \ldots, 8$. On le retarde de $t$ cases : $t$ zéros entrent à gauche, et les $t$ dernières valeurs sortent à droite. Prédis, **avant d'exécuter quoi que ce soit** :
- a) pour $t = 3$, de combien de positions de sortie la sortie est décalée (un entier) ;
- b) la même chose pour $t = 6$ ;
- c) vrai ou faux : pour $t = 1$, la sortie est un décalage de la sortie d'origine ;
- d) la même question pour $t = 2$ ;
- e) les 9 valeurs de la sortie pour $t = 1$, calculées à la main (une liste de 9 entiers).

L'expérience calcule, avec ta `convolve` (une somme glissante est une convolution valide par le noyau retourné, dont on garde une valeur sur 3), la sortie pour $t = 0, 1, 2, 3$ et $6$, et dit pour chacune si c'est un décalage de la sortie d'origine.

Écris tes cinq prédictions, vérifie-les, puis lance l'expérience.

In [ ]:
X_24 = np.zeros(30)
X_24[7:10] = [2.0, 5.0, -1.0]                       # a signal of 30 values, zero except x7, x8 and x9
W_24 = np.array([1.0, -1.0, 2.0, 1.0])              # the weights of the sliding sum
STEP_24 = 3                                          # valid mode: 9 positions

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M21_24a = ...  # a) an integer, written BEFORE the experiment: for t = 3, by how many positions the output moves
prediction_M21_24b = ...  # b) an integer: the same for t = 6
prediction_M21_24c = ...  # c) True or False: for t = 1, the output is a shift of the original output
prediction_M21_24d = ...  # d) True or False: the same for t = 2
prediction_M21_24e = ...  # e) a list of 9 integers, computed by hand: the output for t = 1

In [ ]:
for letter, prediction in zip("abcde", [prediction_M21_24a, prediction_M21_24b, prediction_M21_24c,
                                          prediction_M21_24d, prediction_M21_24e]):
    wbm.check(f"M21.24{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes cinq prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M21_24a, prediction_M21_24b, prediction_M21_24c, prediction_M21_24d, prediction_M21_24e]):
    print("⏳ Écris d'abord tes cinq prédictions, puis exécute à nouveau.")
else:
    with checked("M21.24"):
        convolve_24 = mymath.convolution.convolve

        def strided_24(x):
            """The sliding sum of weights W_24 with the step STEP_24: a valid convolution by the flipped kernel, one value
            in STEP_24."""
            return convolve_24(x, W_24[::-1], "valid")[::STEP_24]

        def delayed_24(x, t):
            """x delayed by t places: t zeros enter on the left, the last t values leave on the right."""
            return np.concatenate([np.zeros(t), x[:len(x) - t]])

        def shift_of_24(y, y0):
            """The m ≥ 0 such that y is y0 moved m places to the right, or None."""
            return next((m for m in range(len(y0)) if np.allclose(y[m:], y0[:len(y0) - m]) and np.allclose(y[:m], 0)),
                        None)

        y0_24 = strided_24(X_24)
        for t_24 in (0, 1, 2, 3, 6):
            y_24 = strided_24(delayed_24(X_24, t_24))
            m_24 = shift_of_24(y_24, y0_24)
            print(f"t = {t_24} : {np.round(y_24).astype(int).tolist()}  →  "
                  + (f"la sortie d'origine décalée de {m_24} position(s)" if m_24 is not None
                     else "pas un décalage de la sortie d'origine"))

**Après l'expérience**, dans ta copie : quels décalages de l'entrée décalent la sortie, et de combien ? Pour $t = 1$, quelles valeurs du signal lit chaque fenêtre ? Que deviendrait l'expérience avec un pas de 1 ? Pourquoi dit-on qu'un filtrage avec pas n'est équivariant que pour les décalages multiples du pas ?

### Ex M21.25 — Le coût d'un filtrage direct ou séparable, selon la taille du noyau : mesurer, puis comparer à k² et 2k 🔬 ★★ ⏱️ 30 min
**Objectif :** mesurer le coût d'un filtrage direct et d'un filtrage séparable selon la taille du noyau, et le comparer aux comptes $k^2$ et $2k$ de la fiche.  
**Prérequis :** fiche §6 ; Ex M21.8 · **Parcours :** C

Une grille de $360 \times 360$ valeurs tirées au hasard (`GRID_25`, graine 5535) est filtrée, en mode valide, par des noyaux de rang 1, $\mathbf{K} = \mathbf{a}\mathbf{b}^\top$ (fiche §6), de taille $k \times k$ pour $k = 3, 5, 9, 15, 25$ (`KS_25` ; les facteurs $\mathbf{a}$ et $\mathbf{b}$ de chaque $k$ sont dans `FACTORS_25[k]`). Écris :
- `direct_25(X, a, b)` : la somme glissante valide de la grille $\mathbf{X}$ par $\mathbf{K} = \mathbf{a}\mathbf{b}^\top$, qui ajoute les $k^2$ tranches de $\mathbf{X}$ décalées de $(u, v)$, chacune multipliée par $K_{u,v}$ ;
- `separable_25(X, a, b)` : la même chose en deux passes, $k$ tranches le long des colonnes (décalées de $u$ lignes) multipliées par $a_u$, puis $k$ tranches du résultat le long des lignes (décalées de $v$ colonnes) multipliées par $b_v$ ;
- `draw_costs_25(ax_time, ax_ratio, ks, direct_times, separable_times)`, qui dessine dans les deux axes reçus (sans en ouvrir d'autre) : dans `ax_time`, les deux temps contre $k$, une ligne avec des points pour chacun, les deux échelles logarithmiques ; dans `ax_ratio`, le rapport des temps (direct / séparable) contre $k$, et la droite $\frac k2$ ; des légendes et des titres.

Puis mesure, avec `best_time_25` (fournie : le meilleur de 3 essais, `time.perf_counter`), `direct_times_25` et `separable_times_25`, les temps de tes deux fonctions sur `GRID_25`, un par $k$ de `KS_25`. Réponds :
- a) le nombre de multiplications par valeur de sortie pour $k = 25$, selon les comptes de la fiche ($k_hk_w$ en direct, environ $k_h + k_w$ en deux passes), dans cet ordre (une liste de 2 entiers) ;
- b) le rapport mesuré des temps pour $k = 25$ est le plus proche, en ordre de grandeur (sur une échelle logarithmique), de : (A) 2 ; (B) 12,5 ; (C) 625 (une lettre) ;
- c) vrai ou faux : pour $k = 25$, les deux sorties coïncident à $10^{-9}$ près (`np.allclose(..., rtol=0, atol=1e-9)`).

La vérification essaie tes deux filtrages sur des grilles que tu ne vois pas (comparés à `scipy.signal.correlate2d`), puis appelle `draw_costs_25` et regarde le dessin, en s'arrêtant au premier ❌. (La transformée de Fourier, qui accélère encore les très grands noyaux, n'est qu'évoquée dans la fiche, §4.)

In [ ]:
rng_25 = np.random.default_rng(5535)
GRID_25 = rng_25.standard_normal((360, 360))                    # a grid of 360 × 360 random values
KS_25 = [3, 5, 9, 15, 25]
FACTORS_25 = {k: (rng_25.standard_normal(k), rng_25.standard_normal(k)) for k in KS_25}   # K = outer(a, b) for each k


def best_time_25(function, *args, repeat=3):
    """The best of `repeat` runs of function(*args), in seconds (time.perf_counter)."""
    best = math.inf
    for _ in range(repeat):
        start = time.perf_counter()
        function(*args)
        best = min(best, time.perf_counter() - start)
    return best

In [ ]:
def direct_25(X, a, b):
    """The valid sliding sum of the grid X by the k × k kernel K = outer(a, b): the sum of the k² slices of X shifted
    by (u, v), each multiplied by K[u, v]."""
    raise NotImplementedError("direct_25 n'est pas encore écrite (04_indices.md si tu bloques)")


def separable_25(X, a, b):
    """The same in two passes: k slices along the columns (shifted by u rows) multiplied by a[u], then k slices of
    the result along the rows (shifted by v columns) multiplied by b[v]."""
    raise NotImplementedError("separable_25 n'est pas encore écrite (04_indices.md si tu bloques)")


def draw_costs_25(ax_time, ax_ratio, ks, direct_times, separable_times):
    """In ax_time: the two times against k (one line with points each), both scales logarithmic; in ax_ratio: the
    ratio direct / separable against k, and the line k / 2. No other axes."""
    raise NotImplementedError("draw_costs_25 n'est pas encore écrite (04_indices.md si tu bloques)")


direct_times_25 = separable_times_25 = answer_M21_25a = answer_M21_25b = answer_M21_25c = ...
with wbm.attempt("M21.25"):          # your functions: ⏳ while they are not written
    direct_times_25 = ...     # the 5 best times of direct_25 on GRID_25, one per k of KS_25 (best_time_25)
    separable_times_25 = ...  # the same for separable_25
    answer_M21_25a = ...  # a) a list of 2 integers: multiplications per output value for k = 25, direct then two passes
    answer_M21_25b = ...  # b) "A", "B" or "C": your measured ratio for k = 25 is closest, on a log scale, to (A) 2, (B) 12.5, (C) 625
    answer_M21_25c = ...  # c) True or False: for k = 25, the two outputs coincide to 1e-9

In [ ]:
wbm.check("M21.25a", answer_M21_25a)
wbm.check("M21.25b", answer_M21_25b)
wbm.check("M21.25c", answer_M21_25c, computed=True)

with checked("M21.25"):
    results_25 = []

    def report_25(ok, success, failure):
        """One ✅ line, or one ❌ line that stops the check there (the figure is shown when it exists)."""
        results_25.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))
        if not ok:
            raise StopCheck

    rng_c25 = np.random.default_rng(5525)
    for name_c25, function_c25 in (("direct_25", direct_25), ("separable_25", separable_25)):
        wrong_c25 = []
        for shape_c25, k_c25 in (((12, 10), 3), ((9, 14), 4), ((8, 8), 8), ((7, 11), 1)):
            X_c25 = rng_c25.standard_normal(shape_c25)
            a_c25, b_c25 = rng_c25.standard_normal(k_c25), rng_c25.standard_normal(k_c25)
            want_c25 = scipy.signal.correlate2d(X_c25, np.outer(a_c25, b_c25), "valid")
            got_c25 = function_c25(X_c25.copy(), a_c25.copy(), b_c25.copy())
            where_c25 = f"une grille {shape_c25[0]} × {shape_c25[1]}, k = {k_c25}"
            if got_c25 is None or np.shape(got_c25) != want_c25.shape:
                wrong_c25.append(f"{where_c25} : attendu la forme {want_c25.shape}, reçu "
                                 + ("None" if got_c25 is None else f"la forme {np.shape(got_c25)}"))
            elif not np.allclose(np.asarray(got_c25, dtype=float), want_c25):
                wrong_c25.append(f"{where_c25} : la bonne forme, mais un écart de "
                                 f"{np.max(np.abs(np.asarray(got_c25, dtype=float) - want_c25)):.3g} avec "
                                 "scipy.signal.correlate2d")
        report_25(not wrong_c25, f"{name_c25} : juste sur quatre grilles que tu ne vois pas (comparé à "
                  "scipy.signal.correlate2d)", f"{name_c25}, " + " ; ".join(wrong_c25[:2]))
    if not filled(direct_times_25, separable_times_25):
        print("⏳ Ex M21.25 : la figure attend tes temps, direct_times_25 et separable_times_25.")
        raise StopCheck
    times_c25 = [np.asarray(times, dtype=float).ravel() for times in (direct_times_25, separable_times_25)]
    report_25(all(t.shape == (len(KS_25),) and np.all(t > 0) for t in times_c25),
              "tes temps : cinq nombres positifs par fonction, un par k de KS_25",
              "direct_times_25 et separable_times_25 doivent avoir un temps positif par k de KS_25")
    fig_25, (ax_time_25, ax_ratio_25) = plt.subplots(1, 2, figsize=(12, 4.2))
    try:
        opened_25 = drawn(draw_costs_25, ax_time_25, ax_ratio_25, list(KS_25), times_c25[0].tolist(),
                          times_c25[1].tolist())

        def through_25(ax, ys):
            """True when one line of ax passes by all the points (k, y) of KS_25 and ys."""
            return any(np.asarray(line.get_xdata(), dtype=float).shape == (len(KS_25),)
                       and np.allclose(np.asarray(line.get_xdata(), dtype=float), KS_25)
                       and np.allclose(np.asarray(line.get_ydata(), dtype=float), ys, rtol=1e-9) for line in ax.lines)

        report_25(not opened_25, "draw_costs_25 dessine dans les axes reçus",
                  "draw_costs_25 doit dessiner dans les deux axes reçus, sans ouvrir de figure")
        report_25(through_25(ax_time_25, times_c25[0]) and through_25(ax_time_25, times_c25[1]),
                  "les deux temps sont tracés contre k", "il manque, dans le premier axe, une courbe des temps contre k "
                  "(une par fonction, une valeur par k de KS_25)")
        report_25(ax_time_25.get_xscale() == "log" and ax_time_25.get_yscale() == "log",
                  "les temps sont en échelles logarithmiques", "mets les deux échelles du premier axe en logarithmique "
                  "(ax.set_xscale(\"log\"), ax.set_yscale(\"log\"))")
        report_25(through_25(ax_ratio_25, times_c25[0] / times_c25[1]) and through_25(ax_ratio_25, np.array(KS_25) / 2),
                  "le rapport des temps est tracé à côté de k/2", "il manque, dans le second axe, le rapport des temps "
                  "(direct / séparable) contre k, ou la droite k/2")
    except StopCheck:
        plt.show()
        raise
    except BaseException:
        plt.close(fig_25)
        raise
    plt.show()
    verdict("M21.25", all(results_25), "tes deux filtrages sont justes, et leurs coûts sont dessinés.", "")

**Après la figure**, dans ta copie : pourquoi le temps suit-il le nombre de tranches ? Pour quels $k$ le séparable vaut-il la peine ? Pourquoi la passe intermédiaire a-t-elle un peu plus de valeurs que la sortie, et que devient le compte « environ $2k$ » pour une très grande grille ? Que ferais-tu pour un noyau de rang 2 ?

### Ex M21.26 — La carte des recouvrements d'une convolution transposée : un damier ou pas, selon k et s 🎨 ★ ⏱️ 15 min
**Objectif :** dessiner, en chaque case de la sortie d'une convolution transposée en deux dimensions, le nombre de tampons qui la couvrent, et voir quand apparaît un damier.  
**Prérequis :** fiche §5 ; Ex M21.7 · **Parcours :** C, E

Une convolution transposée de pas $s$ imprime, pour chaque valeur d'entrée $y_i$, un tampon $y_i\,\mathbf{w}$ à partir de la case $s\,i$ (fiche §5) ; une case de la sortie reçoit un tampon de chaque $i$ tel que $0 \le j - s\,i \le k - 1$. En deux dimensions, avec une entrée de $m \times m$ valeurs et des noyaux $k \times k$ de pas $s$ selon les deux axes, le nombre de tampons qui couvrent la case $(j, j')$ est le **produit** des comptes des deux axes. Écris :
- `coverage_26(m, k, s)` : en une dimension, le nombre de tampons qui couvrent chaque case, un tableau d'entiers de longueur $(m - 1)s + k$ ;
- `coverage_map_26(m, k, s)` : la carte en deux dimensions, le tableau des produits des comptes des deux axes ;
- `draw_maps_26(fig, axes, m, pairs)` : une carte par couple $(k, s)$ de `pairs`, dans les axes de `axes`, dans cet ordre : l'image de `coverage_map_26(m, k, s)` (`ax.imshow`), sa barre de couleurs (`fig.colorbar(image, ax=ax)`) et un titre qui donne $(k, s)$, avec `ax.grid(False)` pour que la grille du style ne coupe pas les cases ; sans ouvrir d'autre figure.

La cellule fournie donne $m = 6$ (`M_26`) et les couples $(k, s) = (5, 2)$, $(6, 2)$, $(7, 3)$ et $(9, 3)$ (`PAIRS_26`). La vérification compare ta carte à des tampons posés un par un, sur ces quatre couples et sur des cas que tu ne vois pas, puis appelle `draw_maps_26` et regarde le dessin, en s'arrêtant au premier ❌ : chaque carte, sa barre de couleurs et son titre.

In [ ]:
M_26 = 6                                                   # a 6 × 6 input
PAIRS_26 = [(5, 2), (6, 2), (7, 3), (9, 3)]                # (k, s): k × k kernels, step s along both axes

In [ ]:
def coverage_26(m, k, s):
    """The number of stamps that cover each cell of a 1-D transposed convolution: m inputs, a kernel of length k,
    the step s, no padding. A 1-D array of integers, of length (m - 1) s + k."""
    raise NotImplementedError("coverage_26 n'est pas encore écrite (04_indices.md si tu bloques)")


def coverage_map_26(m, k, s):
    """The same for an m × m input, k × k kernels and the step s along both axes: the 2-D array of the products of
    the counts of the two axes."""
    raise NotImplementedError("coverage_map_26 n'est pas encore écrite (04_indices.md si tu bloques)")


def draw_maps_26(fig, axes, m, pairs):
    """One map per (k, s) of pairs, in the axes of `axes` (in this order): the image of coverage_map_26(m, k, s),
    its colour bar and a title that gives (k, s). No other figure."""
    raise NotImplementedError("draw_maps_26 n'est pas encore écrite (04_indices.md si tu bloques)")

In [ ]:
with checked("M21.26"):
    results_26 = []

    def report_26(ok, success, failure):
        """One ✅ line, or one ❌ line that stops the check there."""
        results_26.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))
        if not ok:
            raise StopCheck

    def stamps_c26(m, k, s):
        """The stamps on each cell of a 1-D transposed convolution, put one by one."""
        counts = np.zeros((m - 1) * s + k, dtype=int)
        for i in range(m):
            counts[s * i:s * i + k] += 1
        return counts

    cases_c26 = [(M_26, k, s) for k, s in PAIRS_26] + [(3, 4, 4), (5, 2, 3), (4, 1, 1), (2, 7, 5)]
    wrong_c26 = []
    for m_c26, k_c26, s_c26 in cases_c26:
        want_c26 = np.outer(stamps_c26(m_c26, k_c26, s_c26), stamps_c26(m_c26, k_c26, s_c26))
        got_c26 = coverage_map_26(m_c26, k_c26, s_c26)
        if got_c26 is None or np.shape(got_c26) != want_c26.shape or not np.array_equal(got_c26, want_c26):
            wrong_c26.append(f"m = {m_c26}, k = {k_c26}, s = {s_c26} : attendu une carte {want_c26.shape}, reçu "
                             + ("None" if got_c26 is None else f"une carte {np.shape(got_c26)}"
                                + (" aux comptes différents" if np.shape(got_c26) == want_c26.shape else "")))
    report_26(not wrong_c26, f"coverage_map_26 : juste sur {len(cases_c26)} cas (tampons posés un par un)",
              "coverage_map_26, " + " ; ".join(wrong_c26[:2]))
    fig_26, axes_26 = plt.subplots(1, len(PAIRS_26), figsize=(16, 3.9))
    try:
        opened_26 = drawn(draw_maps_26, fig_26, list(axes_26), M_26, list(PAIRS_26))
        report_26(not opened_26, "draw_maps_26 dessine dans la figure reçue",
                  "draw_maps_26 doit dessiner dans les axes reçus, sans ouvrir de figure")
        for ax_c26, (k_c26, s_c26) in zip(axes_26, PAIRS_26):
            want_c26 = np.outer(stamps_c26(M_26, k_c26, s_c26), stamps_c26(M_26, k_c26, s_c26))
            images_c26 = [image for image in ax_c26.images if np.shape(image.get_array()) == want_c26.shape
                          and np.array_equal(np.asarray(image.get_array()), want_c26)]
            report_26(images_c26, f"(k, s) = ({k_c26}, {s_c26}) : la carte est dessinée",
                      f"il manque, dans l'axe de (k, s) = ({k_c26}, {s_c26}), l'image de coverage_map_26 "
                      f"({want_c26.shape[0]} × {want_c26.shape[1]} cases ; ax.imshow)")
            report_26(images_c26[0].colorbar is not None, "… avec sa barre de couleurs",
                      f"il manque la barre de couleurs de (k, s) = ({k_c26}, {s_c26}) (fig.colorbar(image, ax=ax))")
            numbers_c26 = [int(n) for n in re.findall(r"\d+", ax_c26.get_title())]
            report_26(k_c26 in numbers_c26 and s_c26 in numbers_c26, "… et un titre qui donne (k, s)",
                      f"le titre de l'axe de (k, s) = ({k_c26}, {s_c26}) doit donner k et s")
    except StopCheck:
        plt.show()
        raise
    except BaseException:
        plt.close(fig_26)
        raise
    plt.show()
    verdict("M21.26", all(results_26), "les quatre cartes des recouvrements sont dessinées.", "")

**Après la figure**, dans ta copie : pour quels couples la carte est-elle uniforme à l'intérieur, et quelle valeur y prend-elle ? Pourquoi un damier apparaît-il quand $s$ ne divise pas $k$ ? Quelles valeurs prend la carte à l'intérieur pour $(5, 2)$ ? Pourquoi les bords reçoivent-ils moins de tampons dans tous les cas ?

## Partie C · Déboguer, tester

Fiche §2, §3, §5 et §6. Trois erreurs classiques qui ne lèvent aucune erreur Python ; un test de propriété pour la matrice de Toeplitz. La vérification de M21.28 s'appuie sur la référence (`ref_conv`), pour ne pas dépendre de ton code.

In [ ]:
ref_conv = wbm.load_mymath("ref", alias="mymath_ref").convolution   # the reference: the check of M21.28

### Ex M21.27 — Chasse à l'erreur : un noyau non retourné, des recouvrements supposés réguliers, un noyau de rang 2 « séparé » 🐛 ★ ⏱️ 25 min
**Objectif :** reconnaître trois erreurs classiques qui ne lèvent aucune erreur Python et passent une vérification mal choisie, puis les corriger.  
**Prérequis :** fiche §1, §5 et §6 ; Ex M21.7 et M21.22 · **Parcours :** C, E

Un collègue a écrit trois fonctions (cellule fournie) ; chacune a l'erreur que nomme le titre, aucune ne lève d'erreur Python, et chacune a passé la vérification qu'il a faite :
1. `poly_product(p, q)` doit rendre les coefficients du produit de deux polynômes, du terme constant au terme de plus haut degré ; elle renvoie `np.correlate(p, q, "full")`. Il l'a vérifiée sur $(1 + 2X)(2 + 5X + 2X^2)$, dont le second facteur a des coefficients palindromes, et l'applique à $(1 + 2X)(3 - X + 4X^2)$ (`P_27`, `Q_27`).
2. `n_covers(k, s)` doit rendre le nombre de tampons qui couvrent une case intérieure d'une convolution transposée de noyau de longueur $k$ et de pas $s$ (fiche §5) ; elle renvoie `k // s`. Il l'a vérifiée sur $(k, s) = (6, 3)$, $(8, 2)$ et $(12, 4)$.
3. `separate(K)` doit rendre les deux facteurs d'un noyau séparable ; elle rend la première colonne de $\mathbf{K}$ et sa première ligne divisée par $K_{0,0}$, sans regarder le rang. Il l'a vérifiée sur un produit extérieur, et l'applique au noyau $\begin{pmatrix} 2 & 1 & 3 \\ 4 & 2 & 5 \\ 2 & 1 & 3 \end{pmatrix}$ (`K_27`).

À faire : écris `poly_product_fixed(p, q)`, `n_covers_fixed(j, k, s, m)` (le nombre de tampons qui couvrent la case $j$ d'une convolution transposée de $m$ valeurs d'entrée, de noyau de longueur $k$ et de pas $s$ : le nombre de $i \in \{0, \ldots, m - 1\}$ tels que $0 \le j - s\,i \le k - 1$) et `separate_fixed(K)` (qui rend les deux facteurs d'un noyau de rang 1, et `None` sinon), puis réponds :
- a) les coefficients de $(1 + 2X)(3 - X + 4X^2)$, avec `poly_product_fixed` (la valeur que donne le calcul) ;
- b) pour une convolution transposée de noyau de longueur $k = 8$, de pas $s = 3$ et de $m = 6$ valeurs d'entrée, les nombres de tampons qui couvrent les cases 9, 10 et 11, avec `n_covers_fixed` (une liste de 3 entiers) ;
- c) le plus grand écart, en valeur absolue, entre `K_27` et le produit extérieur des deux facteurs que rend le `separate` du collègue (la valeur que donne le calcul) ;
- d) vrai ou faux : ta `separable_factors` (M21.22) rend `None` pour ce noyau ;
- e) une question à part, sur le mode « même taille » d'un noyau de longueur paire, où NumPy et PyTorch diffèrent d'une case : pour la somme glissante de poids $(2, -3)$ et $\mathbf{x} = (6, 2, 7, 1, 8, 3)$ (`W_27`, `X_27`), ce que rend `np.convolve` en mode même taille, avec les poids retournés (la valeur que donne le calcul).

La vérification essaie tes corrections sur des cas que tu ne vois pas : des polynômes aux coefficients non palindromes, toutes les cases de convolutions transposées où $s$ divise $k$ ou pas, des noyaux de rang 1 (dont un avec $K_{0,0} = 0$) et de rang 2.

In [ ]:
P_27, Q_27 = np.array([1.0, 2.0]), np.array([3.0, -1.0, 4.0])      # (1 + 2X)(3 − X + 4X²)
K_27 = np.array([[2.0, 1.0, 3.0], [4.0, 2.0, 5.0], [2.0, 1.0, 3.0]])
X_27, W_27 = np.array([6.0, 2.0, 7.0, 1.0, 8.0, 3.0]), np.array([2.0, -3.0])


# The colleague's code
def poly_product(p, q):
    """The coefficients of the product of two polynomials, from the constant term to the highest degree."""
    return np.correlate(p, q, "full")


def n_covers(k, s):
    """The number of stamps that cover an inner cell of a transposed convolution: kernel of length k, step s."""
    return k // s


def separate(K):
    """The two factors (u, v) of a separable kernel, K = outer(u, v)."""
    K = np.asarray(K, dtype=float)
    return K[:, 0].copy(), K[0, :] / K[0, 0]


# The colleague's checks: all pass
print("(1) (1 + 2X)(2 + 5X + 2X²) :", poly_product([1, 2], [2, 5, 2]).tolist(), "(attendu [2, 9, 12, 4])")
print("(2) n_covers : (k, s) = (6, 3) →", n_covers(6, 3), "; (8, 2) →", n_covers(8, 2), "; (12, 4) →", n_covers(12, 4),
      "(comptés à la main sur une case intérieure : k/s)")
u_c27, v_c27 = separate(np.outer([1, 2], [3, 1, 2]))
print("(3) un produit extérieur reconstruit :", bool(np.allclose(np.outer(u_c27, v_c27), np.outer([1, 2], [3, 1, 2]))))

In [ ]:
def poly_product_fixed(p, q):
    """The coefficients of the product of two polynomials, from the constant term to the highest degree."""
    raise NotImplementedError("poly_product_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def n_covers_fixed(j, k, s, m):
    """The number of stamps that cover the cell j of a transposed convolution of m inputs, kernel of length k, step s:
    the number of i in {0, ..., m - 1} such that 0 <= j - s i <= k - 1."""
    raise NotImplementedError("n_covers_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def separate_fixed(K):
    """The two factors (u, v) of K = outer(u, v) when K has rank 1; None otherwise."""
    raise NotImplementedError("separate_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M21_27a = answer_M21_27b = answer_M21_27c = answer_M21_27d = answer_M21_27e = ...
with wbm.attempt("M21.27"):          # your three functions: ⏳ while they are not written
    answer_M21_27a = ...  # a) a list of 4 numbers, as computed: poly_product_fixed(P_27, Q_27)
    answer_M21_27b = ...  # b) a list of 3 integers: the stamps on the cells 9, 10 and 11 (k = 8, s = 3, m = 6)
    answer_M21_27c = ...  # c) a number, as computed: the largest |K_27 - outer(u, v)|, (u, v) from the colleague's separate
    answer_M21_27d = ...  # d) True or False: your separable_factors(K_27) is None
    answer_M21_27e = ...  # e) a list of 6 numbers, as computed: np.convolve(X_27, W_27[::-1], "same")

In [ ]:
for letter, answer in zip("abcde", [answer_M21_27a, answer_M21_27b, answer_M21_27c, answer_M21_27d, answer_M21_27e]):
    wbm.check(f"M21.27{letter}", answer, computed=True)

results_27, parts_27 = [], []


def report_27(ok, success, failure):
    results_27.append(bool(ok))
    print(("✅ " + success) if ok else ("❌ " + failure))


def call_27(part, name, function, *args):
    """function(*args); an error of the learner's code becomes one ❌ line that names the function (the check of this
    part stops there, the final verdict stays)."""
    try:
        return function(*args)
    except (NotImplementedError, MymathMissingError):
        raise
    except Exception as exc:  # noqa: BLE001 - the learner's error, reported with the name of the function
        report_27(False, "", f"({part}) {name} : ton code lève {type(exc).__name__} : {str(exc)[:200]}")
        parts_27.append(part)
        raise StopCheck from None


rng_c27 = np.random.default_rng(5521)
with checked("M21.27"):                                       # (1) poly_product_fixed
    wrong_27 = []
    for n_c27, m_c27 in ((2, 4), (4, 2), (3, 3), (5, 1), (1, 3)):
        p_c27 = rng_c27.integers(-5, 6, size=n_c27).astype(float)
        q_c27 = rng_c27.integers(-5, 6, size=m_c27).astype(float)
        q_c27[0], q_c27[-1] = 2.0, -3.0 if m_c27 > 1 else 2.0         # never a palindrome
        want_c27 = np.array([sum(p_c27[i] * q_c27[d - i] for i in range(n_c27) if 0 <= d - i < m_c27)
                             for d in range(n_c27 + m_c27 - 1)])          # the product, expanded term by term
        got_c27 = call_27(1, "poly_product_fixed", poly_product_fixed, p_c27.copy(), q_c27.copy())
        if got_c27 is None or np.shape(got_c27) != want_c27.shape or not np.allclose(got_c27, want_c27):
            wrong_27.append(f"p = {p_c27.astype(int).tolist()}, q = {q_c27.astype(int).tolist()} : attendu "
                            f"{want_c27.astype(int).tolist()}, reçu {shown(got_c27, 3)}"[:260])
    report_27(not wrong_27, "(1) poly_product_fixed : des polynômes aux coefficients non palindromes, de degrés variés "
              "(comparé au développement terme à terme)", "(1) poly_product_fixed : " + " ; ".join(wrong_27[:2]))
    parts_27.append(1)

with checked("M21.27"):                                       # (2) n_covers_fixed
    wrong_27 = []
    for k_c27, s_c27, m_c27 in ((8, 3, 6), (5, 2, 4), (7, 3, 5), (6, 2, 3), (4, 5, 3), (1, 1, 4)):
        stamps_c27 = np.zeros((m_c27 - 1) * s_c27 + k_c27, dtype=int)
        for i_c27 in range(m_c27):
            stamps_c27[s_c27 * i_c27:s_c27 * i_c27 + k_c27] += 1          # the stamps, put one by one
        got_c27 = [call_27(2, "n_covers_fixed", n_covers_fixed, j_c27, k_c27, s_c27, m_c27)
                   for j_c27 in range(len(stamps_c27))]
        if got_c27 != stamps_c27.tolist():
            wrong_27.append(f"(k, s, m) = ({k_c27}, {s_c27}, {m_c27}), toutes les cases : attendu "
                            f"{stamps_c27.tolist()}, reçu {got_c27!r}"[:220])
    report_27(not wrong_27, "(2) n_covers_fixed : toutes les cases de six convolutions transposées, s divisant k ou "
              "pas (tampons posés un par un)", "(2) n_covers_fixed : " + " ; ".join(wrong_27[:2]))
    parts_27.append(2)

with checked("M21.27"):                                       # (3) separate_fixed
    wrong_27 = []
    rank_one_c27 = [np.outer([0.0, 2.0, -1.0], [3.0, 1.0, 2.0]), np.outer([1.5, -2.0], [0.0, 3.0, 1.0, -2.0]),
                    np.outer(rng_c27.normal(size=4), rng_c27.normal(size=3))]
    rank_two_c27 = [np.array([[1.0, 2.0, 1.0], [2.0, 4.0, 3.0]]), rng_c27.normal(size=(3, 3))]
    for kernel_c27 in rank_one_c27:
        got_c27 = call_27(3, "separate_fixed", separate_fixed, kernel_c27.copy())
        if (not isinstance(got_c27, tuple) or len(got_c27) != 2
                or np.shape(np.outer(*got_c27)) != kernel_c27.shape or not np.allclose(np.outer(*got_c27), kernel_c27)):
            wrong_27.append(f"un noyau de rang 1, {np.round(kernel_c27, 2).tolist()} : attendu deux facteurs dont "
                            f"le produit extérieur le redonne, reçu {shown(got_c27, 3)}"[:260])
    for kernel_c27 in rank_two_c27:
        got_c27 = call_27(3, "separate_fixed", separate_fixed, kernel_c27.copy())
        if got_c27 is not None:
            wrong_27.append(f"un noyau de rang 2 ou plus, {np.round(kernel_c27, 2).tolist()} : attendu None"[:160])
    report_27(not wrong_27, "(3) separate_fixed : des noyaux de rang 1 (dont un avec K[0, 0] = 0) et de rang 2 ou plus",
              "(3) separate_fixed : " + " ; ".join(wrong_27[:2]))
    parts_27.append(3)

if len(set(parts_27)) == 3:
    verdict("M21.27", all(results_27), "les trois fonctions corrigées passent tous les cas.",
            "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : pour chaque erreur, pourquoi la vérification du collègue ne la voyait-elle pas (un facteur palindrome, des couples où $s$ divise $k$, un noyau de rang 1), et quelle vérification l'aurait vue ? Écris la carte complète des tampons pour $k = 8$, $s = 3$, $m = 6$. Pour (3), quelle case de `K_27` le produit extérieur du collègue manque-t-il, et que se passerait-il avec $K_{0,0} = 0$ ? Pour e), de quel côté est le zéro en trop dans chaque bibliothèque ?

### Ex M21.28 — Un test de propriété : la matrice de Toeplitz fois x égale la convolution, pour des longueurs tirées au hasard 🛠️ ★ ⏱️ 20 min
**Objectif :** écrire un test de propriété pour `toeplitz_matrix`, et s'assurer qu'il attrape une version fausse du mode même taille.  
**Prérequis :** fiche §3 ; Ex M21.20 · **Parcours :** C

Un **test de propriété** vérifie, sur beaucoup de cas tirés au hasard, une loi que tout résultat juste respecte, sans connaître la réponse de chaque cas (M17, M18). La loi, ici (fiche §3) : pour tout noyau $\mathbf{w}$ de longueur $k$, tout signal $\mathbf{x}$ de longueur $n \ge k$ et chacun des trois modes, `toeplitz_matrix(w, n, mode) @ x` égale `convolve(x, w, mode)`.

Écris `property_test_28(toeplitz_matrix, convolve)`, qui reçoit les deux fonctions à tester :
- elle tire 200 cas avec un générateur `np.random.default_rng(5538)`, créé dans la fonction : une longueur de noyau $k$ de 1 à 6, une longueur de signal $n$ de $k$ à 12, un noyau et un signal d'entiers ; puis elle ajoute les cas limites $k = 1$ et $n = k$ ;
- pour chaque cas et chaque mode (`"full"`, `"valid"`, `"same"`), elle vérifie la loi par un `assert` (avec `np.allclose`), dont le message dit **d'abord** ce qui était attendu (il commence par « expected »), **puis** le cas fautif, écrit `k = …, n = …` avec le mode ;
- elle rend le nombre de cas vérifiés.

La cellule fournie donne une version **fausse** de `toeplitz_matrix`, construite sur la référence (`ref_conv`) : `toeplitz_wrong_same_28`, dont le mode même taille garde les lignes à partir de $\lfloor k/2\rfloor$ au lieu de $\lfloor\frac{k-1}{2}\rfloor$. La vérification lance ton test sur la référence, où il doit passer (elle compte les appels, les modes et les cas limites) ; sur la version fausse, où il doit échouer par un `assert` dont le message montre un noyau de longueur paire ; puis sur **ta** librairie.

In [ ]:
def toeplitz_wrong_same_28(kernel, n, mode="full"):
    """Wrong: in the mode "same", the rows kept start at k // 2 instead of (k - 1) // 2 (built on the reference)."""
    if mode != "same":
        return ref_conv.toeplitz_matrix(kernel, n, mode)
    k = len(kernel)
    return ref_conv.toeplitz_matrix(kernel, n, "full")[k // 2:k // 2 + max(n, k)]

In [ ]:
def property_test_28(toeplitz_matrix, convolve):
    """200 random cases drawn with np.random.default_rng(5538), created inside: k from 1 to 6, n from k to 12, a
    kernel and a signal of integers; then the edge cases k = 1 and n = k. For each case and each mode ("full",
    "valid", "same"), assert that toeplitz_matrix(w, n, mode) @ x equals convolve(x, w, mode), with a message that
    says first what was expected, then the case (k = ..., n = ..., the mode). Return the number of cases checked."""
    raise NotImplementedError("property_test_28 n'est pas encore écrite (04_indices.md si tu bloques)")

In [ ]:
with checked("M21.28"):
    results_28 = []

    def report_28(ok, success, failure):
        results_28.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))

    calls_28 = []

    def counted_28(toeplitz):
        """toeplitz, its calls recorded in calls_28 as (k, n, mode)."""
        def wrapper(kernel, n, mode="full"):
            calls_28.append((len(kernel), int(n), mode))
            return toeplitz(kernel, n, mode)
        return wrapper

    try:
        cases_28 = property_test_28(counted_28(ref_conv.toeplitz_matrix), ref_conv.convolve)
    except AssertionError as exc:
        print(f"❌ Ex M21.28 : ton test échoue sur la référence, qui est juste : {exc}"[:400])
        raise StopCheck from None
    report_28(isinstance(cases_28, int) and cases_28 >= 202 and len(calls_28) >= 3 * 202,
              f"sur la référence, ton test passe : {cases_28} cas, {len(calls_28)} appels de toeplitz_matrix",
              f"sur la référence, ton test rend {cases_28!r} et fait {len(calls_28)} appels de toeplitz_matrix : au "
              "moins 200 cas tirés et les deux cas limites, chacun dans les trois modes, et le nombre de cas rendu")
    report_28({mode for _, _, mode in calls_28} == {"full", "valid", "same"}, "les trois modes sont testés",
              "ton test doit appeler toeplitz_matrix dans les trois modes, \"full\", \"valid\" et \"same\"")
    report_28(any(k == 1 for k, _, _ in calls_28) and any(k == n for k, n, _ in calls_28),
              "les cas limites k = 1 et n = k sont testés", "ajoute les cas limites k = 1 et n = k")
    try:
        property_test_28(toeplitz_wrong_same_28, ref_conv.convolve)
        report_28(False, "", "ton test n'attrape pas toeplitz_wrong_same_28 (son mode même taille garde les lignes "
                  "à partir de ⌊k/2⌋ ; une AssertionError attrapée dans ta fonction ne compte pas : laisse-la sortir)")
    except AssertionError as exc:
        message_28 = str(exc)
        k_28 = re.search(r"\bk\s*=\s*(\d+)", message_28)
        n_28 = re.search(r"\bn\s*=\s*(\d+)", message_28)
        report_28(True, "ton test attrape toeplitz_wrong_same_28 par un assert", "")
        report_28(message_28.strip().lower().startswith(("expected", "attendu")),
                  "son message dit d'abord ce qui était attendu",
                  f"le message doit commencer par ce qui était attendu (« expected … »), reçu : {message_28[:160]!r}")
        report_28(k_28 is not None and n_28 is not None and "same" in message_28,
                  "… puis le cas fautif : k, n et le mode",
                  f"le message doit donner le cas fautif, « k = …, n = … » et le mode, reçu : {message_28[:160]!r}")
        if k_28 is not None:
            report_28(int(k_28.group(1)) % 2 == 0, f"le cas fautif a un noyau de longueur paire, k = {k_28.group(1)}",
                      f"le cas fautif montré a k = {k_28.group(1)} : pour un noyau de longueur impaire, les deux "
                      "versions coïncident ; ton message donne-t-il bien la longueur du noyau ?")
    except Exception as exc:  # noqa: BLE001 - another error than an assert
        report_28(False, "", f"sur toeplitz_wrong_same_28, ton test lève {type(exc).__name__} au lieu d'un assert : "
                  f"{str(exc)[:160]}")
    try:
        property_test_28(mymath.convolution.toeplitz_matrix, mymath.convolution.convolve)
        print("✅ ta librairie (toeplitz_matrix et convolve) passe ton test de propriété")
    except (NotImplementedError, MymathMissingError):
        print("⏳ ta librairie : toeplitz_matrix ou convolve n'est pas encore écrite (M21.19, M21.20)")
    except AssertionError as exc:
        print(f"❌ ta librairie échoue à ton test ({str(exc)[:200]}) : si ton test passe sur la référence, l'erreur est "
              "dans ta toeplitz_matrix ou ta convolve (lance aussi leurs tests, M21.19 et M21.20)")
    verdict("M21.28", all(results_28), "ton test passe sur la référence et attrape la version fausse.",
            "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : pourquoi la version fausse passe-t-elle tous les cas de noyau impair, et les modes plein et valide ? Pourquoi les cas limites $k = 1$ et $n = k$ valent-ils d'être écrits à part ? Pourquoi un test de propriété tire-t-il des **entiers** ?

## Partie D · Défi

Fiche §3. Un noyau inconnu, retrouvé par les moindres carrés à partir d'un signal et de sa version filtrée et bruitée.

### Ex M21.29 — Défi : retrouver un noyau inconnu à partir d'un signal et de sa version filtrée bruitée, par les moindres carrés 🏆 ★★★ ⏱️ 25 min
**Objectif :** retrouver un noyau inconnu par les moindres carrés, à partir d'un signal et de sa version filtrée et bruitée.  
**Prérequis :** fiche §3 ; M09b ; Ex M21.20 · **Parcours :** C

Un capteur a été filtré par une somme glissante de 7 poids inconnus, puis bruité. La cellule fournie tire (graine 5539) les poids secrets, un signal $\mathbf{x}$ de 2 000 valeurs gaussiennes (`X_29`), et la sortie $\mathbf{y} = \mathbf{w} \star \mathbf{x} + 0{,}05\,\boldsymbol{\eta}$ (`Y_29`, 1 994 valeurs, $\boldsymbol{\eta}$ gaussien) ; elle ne garde pas les poids secrets.

La sortie est **linéaire en $\mathbf{w}$** (fiche §3) : retrouver $\mathbf{w}$, c'est minimiser $\lVert\mathbf{A}\mathbf{c} - \mathbf{y}\rVert^2$, un problème de moindres carrés (M09b). Écris `estimate_kernel_29(x, y, k)`, l'estimation des $k$ poids d'une somme glissante valide $\mathbf{y} \approx \mathbf{w} \star \mathbf{x}$, avec ta `toeplitz_matrix` et `np.linalg.lstsq` : les lignes $k - 1$ à $n - 1$ de la matrice pleine de $\mathbf{x}$, pris comme noyau, pour des signaux de longueur $k$, envoient $\tilde{\mathbf{w}}$ (le noyau retourné) sur $\mathbf{w} \star \mathbf{x}$ (tu peux aussi prendre la matrice des fenêtres $\mathbf{X}_{\text{f}}$ de la fiche §3, qui envoie $\mathbf{w}$ sur $\mathbf{w} \star \mathbf{x}$). Puis :
- `w_hat_29` : ton estimation des 7 poids, à partir de `X_29` et `Y_29`.

**Critère de réussite :** a) ton estimation, arrondie à 2 décimales, redonne exactement les 7 poids secrets (une liste de 7 nombres). Puis :
- b) l'écart type du bruit estimé par $\frac{\lVert\mathbf{y} - \mathbf{X}_{\text{f}}\hat{\mathbf{w}}\rVert}{\sqrt{1\,994 - 7}}$ (la valeur que donne le calcul) ;
- c) le plus petit nombre $m$ de valeurs de $\mathbf{y}$, parmi 10, 20, 50, 100, 200 et 500 (`M_CHOICES_29`), pour lequel l'estimation faite avec les $m$ premières valeurs de $\mathbf{y}$ (et leurs fenêtres) a tous ses poids à moins de 0,025 des poids secrets (ceux de a), une fois a) juste).

La vérification essaie aussi `estimate_kernel_29` sur des noyaux que tu ne vois pas. Dans ta copie : pourquoi la matrice de Toeplitz de $\mathbf{x}$ donne-t-elle le noyau **retourné** ? Comment l'erreur de c) décroît-elle avec $m$ ? Pourquoi divise-t-on par $\sqrt{1\,994 - 7}$ en b), et non par $\sqrt{1\,994}$ ?

In [ ]:
def _sensor_29():
    """The filtered and noisy sensor (seed 5539): 7 secret weights, x of 2 000 Gaussian values, y = w ⋆ x + 0.05 η."""
    rng = np.random.default_rng(5539)
    secret = np.round(rng.uniform(-1, 1, 7), 2)
    x = rng.standard_normal(2000)
    y = np.correlate(x, secret, "valid") + 0.05 * rng.standard_normal(2000 - 7 + 1)
    return x, y


X_29, Y_29 = _sensor_29()                                  # the secret weights are not kept
M_CHOICES_29 = [10, 20, 50, 100, 200, 500]
print("x :", X_29.shape, "; y :", Y_29.shape)

In [ ]:
def estimate_kernel_29(x, y, k):
    """The least-squares estimate of the k weights w of a valid sliding sum y ≈ w ⋆ x (len(y) = len(x) - k + 1),
    with your toeplitz_matrix and np.linalg.lstsq."""
    raise NotImplementedError("estimate_kernel_29 n'est pas encore écrite (04_indices.md si tu bloques)")


w_hat_29 = answer_M21_29a = answer_M21_29b = answer_M21_29c = ...
with wbm.attempt("M21.29"):          # estimate_kernel_29 and your toeplitz_matrix: ⏳ while they are not written
    w_hat_29 = ...        # your estimate of the 7 weights, from X_29 and Y_29
    answer_M21_29a = ...  # a) a list of 7 numbers: w_hat_29 rounded to 2 decimals
    answer_M21_29b = ...  # b) a number, as computed: the estimated standard deviation of the noise
    answer_M21_29c = ...  # c) an integer of M_CHOICES_29: the smallest m whose estimate is within 0.025 of a)

In [ ]:
right_29 = wbm.check("M21.29a", answer_M21_29a)
wbm.check("M21.29b", answer_M21_29b, computed=True)
wbm.check("M21.29c", answer_M21_29c)
if filled(answer_M21_29a):
    verdict("M21.29", right_29, "critère atteint : ton estimation redonne les 7 poids secrets.",
            "critère pas encore atteint : ton estimation, arrondie à 2 décimales, ne redonne pas les poids secrets.")

with checked("M21.29"):
    rng_c29 = np.random.default_rng(5529)
    wrong_29 = []
    for n_c29, k_c29, noise_c29 in ((60, 3, 0.0), (400, 5, 0.01), (30, 1, 0.0)):
        w_c29 = np.round(rng_c29.uniform(-2, 2, k_c29), 2)
        x_c29 = rng_c29.standard_normal(n_c29)
        y_c29 = np.correlate(x_c29, w_c29, "valid") + noise_c29 * rng_c29.standard_normal(n_c29 - k_c29 + 1)
        got_c29 = estimate_kernel_29(x_c29.copy(), y_c29.copy(), k_c29)
        tolerance_c29 = 1e-9 if noise_c29 == 0 else 0.01
        if got_c29 is None or np.shape(got_c29) != (k_c29,) or not np.allclose(got_c29, w_c29, rtol=0, atol=tolerance_c29):
            wrong_29.append(f"{k_c29} poids, {n_c29} valeurs, bruit {noise_c29} : attendu {w_c29.tolist()} à "
                            f"{tolerance_c29} près, reçu {shown(got_c29)}"[:260])
    verdict("M21.29", not wrong_29, "estimate_kernel_29 retrouve trois noyaux que tu ne vois pas (dont deux sans bruit, "
            "exactement).", "estimate_kernel_29, " + " ; ".join(wrong_29[:2]))

**Dans ta copie** : trace l'erreur maximale contre $m$ en échelles logarithmiques ; quelle pente vois-tu ? Que deviendrait le problème si l'on ne connaissait pas $k$ ? Pourquoi un filtrage par une couche apprise, dans workbookIA 21, se règle-t-il par une autre méthode, alors que celui-ci se résout en une ligne ?

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer une convolution et une corrélation croisée dans les trois modes, et passer de l'une à l'autre par le noyau retourné ?
2. Sais-tu écrire la matrice de Toeplitz d'une convolution, lire son rang, son noyau et sa transposée ?
3. Sais-tu dire ce qu'un décalage fait à un filtrage, avec ou sans pas, et reconnaître une convolution circulaire ?
4. Sais-tu compter les positions d'une fenêtre, la taille d'une convolution transposée, les paramètres et le coût d'un filtrage à plusieurs canaux ?
5. Sais-tu composer des convolutions, calculer un champ récepteur, séparer un noyau de rang 1, et trouver le pire cas d'une perturbation bornée ?

**La suite** : le chapitre 21 de workbookIA, qui fait glisser des noyaux appris sur des images, règle leur pas, leur padding et leur dilatation, compte paramètres et opérations, calcule des champs récepteurs et agrandit des cartes par des convolutions transposées ; puis M22 (récurrences vectorielles, rayon spectral et normes de matrices).